# 01 — Corpus Collection
Walks the repository and collects every source of ARO knowledge into a single
manifest. Calls the `aro` CLI to capture live grammar, action, and example data.

**Output:** `../data/01_corpus/manifest.json`

In [22]:
import os
import json
import subprocess
from pathlib import Path

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

ARO_ROOT    = (SCRIPT_DIR / '../../').resolve()
ARO_APP_DIR = (SCRIPT_DIR / '../../../ARO-Application/').resolve()
DATA_DIR    = SCRIPT_DIR / '../data/01_corpus'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'ARO root:        {ARO_ROOT}')
print(f'ARO-Application: {ARO_APP_DIR}  (exists: {ARO_APP_DIR.exists()})')
print(f'Output:          {DATA_DIR}')

ARO root:        /Users/kris/Projects/ARO-App
ARO-Application: /Users/kris/Projects/ARO-Application  (exists: True)
Output:          /Users/kris/Projects/ARO-App/Train/script/../data/01_corpus


## Check aro CLI

In [23]:
def run_aro(args, timeout=30):
    try:
        r = subprocess.run(['aro'] + args, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip(), r.stderr.strip(), r.returncode
    except FileNotFoundError:
        return '', 'aro binary not found in PATH', -1
    except subprocess.TimeoutExpired:
        return '', 'timeout', -1

out, err, code = run_aro(['--version'])
print(f'aro: {out or err}')

aro: 0.7.1-beta.24-41-gf8ed5764-dirty


## Collect .aro files

In [24]:
aro_files = []
search_roots = [ARO_ROOT / 'Examples']
if ARO_APP_DIR.exists():
    search_roots.append(ARO_APP_DIR)

for root in search_roots:
    found = list(root.rglob('*.aro'))
    print(f'  {root.name}: {len(found)} .aro files')
    aro_files.extend(found)

print(f'Total: {len(aro_files)} .aro files')

  Examples: 120 .aro files
  ARO-Application: 18 .aro files
Total: 138 .aro files


## Collect Proposals, Book, Swift action files

In [25]:
proposals     = sorted((ARO_ROOT / 'Proposals').glob('*.md'))
book_chapters = sorted((ARO_ROOT / 'Book').rglob('*.md')) if (ARO_ROOT / 'Book').exists() else []
swift_actions = sorted((ARO_ROOT / 'Sources' / 'ARORuntime' / 'Actions').rglob('*.swift'))

print(f'Proposals:     {len(proposals)}')
print(f'Book chapters: {len(book_chapters)}')
print(f'Swift actions: {len(swift_actions)}')

Proposals:     59
Book chapters: 121
Swift actions: 33


## Fetch live data from aro CLI

In [26]:
print('Calling aro CLI...')
syntax_out,   _, _ = run_aro(['syntax'])
actions_out,  _, _ = run_aro(['actions'])
examples_out, _, _ = run_aro(['examples'])

print(f'aro syntax:   {len(syntax_out):,} chars')
print(f'aro actions:  {len(actions_out):,} chars')
print(f'aro examples: {len(examples_out):,} chars')

Calling aro CLI...
aro syntax:   78 chars
aro actions:  79 chars
aro examples: 80 chars


## Read static reference docs

In [27]:
extra_docs = {}
for doc_path in [ARO_ROOT / 'README.md', ARO_ROOT / 'CLAUDE.md', ARO_ROOT / 'OVERVIEW.md']:
    if doc_path.exists():
        extra_docs[doc_path.name] = doc_path.read_text()
        print(f'  {doc_path.name}: {len(extra_docs[doc_path.name]):,} chars')

  README.md: 12,448 chars
  CLAUDE.md: 23,628 chars
  OVERVIEW.md: 13,053 chars


## Save manifest

In [28]:
manifest = {
    'aro_files':     [str(f) for f in aro_files],
    'proposals':     [str(f) for f in proposals],
    'book_chapters': [str(f) for f in book_chapters],
    'swift_actions': [str(f) for f in swift_actions],
    'extra_docs':    {k: v[:60_000] for k, v in extra_docs.items()},
    'aro_syntax':    syntax_out,
    'aro_actions':   actions_out,
    'aro_examples':  examples_out,
}

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

total = len(aro_files) + len(proposals) + len(book_chapters) + len(swift_actions)
print(f'Manifest saved: {manifest_path}')
print(f'Total files indexed: {total}')

Manifest saved: /Users/kris/Projects/ARO-App/Train/script/../data/01_corpus/manifest.json
Total files indexed: 351
